<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii"
REPO_DIR = "flyrank-ml-internship-hadii"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 1

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

Loaded: (30000, 44)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Source:** `docs/flyrank-seo-research-march-2026.pdf`, ML Appendix pages 26–27 ("What Predicts Health?") and 28 ("What Predicts Growth?"), plus the Methodology page.

**Finding 1 — "What Predicts Health?" (Random Forest feature importance for health score).**
The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top
drivers of a Random Forest predicting `health_score`, and — to its credit — already discloses that
"the target itself is partly constructed from some of these inputs." The Methodology page confirms
why: Health Score = impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts).

*My methodology question:* if two of the three top-ranked "predictors" (position, impressions) are
literally addends inside the label's own formula, is this importance ranking telling us anything a
reader couldn't get by reading the scoring formula itself? The paper's own honest-claims instinct is
right (it says "descriptive rather than causal"), but I'd push it one step further: this is Leakage
Taxonomy Type 1 (label-derived features) by definition, not just a confound to caveat. The
constructive fix I'd suggest is the same "train-with vs train-without-the-suspect" collapse test I
run on my own model in Section 3 — refit the Random Forest with position/impressions/CTR/scroll
depth removed and show what's left. If nothing meaningful survives, that's worth saying directly
rather than folding into a caveat sentence.

**Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy).**
The model separates growing from declining pages using an 80/20 holdout split (per the Methodology
page: "Random Forest (80/20 split), Logistic Regression (80/20 split)... "), with Content Age
reported as the strongest negative signal.

*My methodology question:* is that 80/20 split a random row-level split, or is it grouped by the
57 brands in the sample? The paper doesn't say. This is exactly the question I had to answer for my
own Week-5 model (Section 2 below) — with 61.8K rows spread across 57 brands, a plain random split lets rows from the same brand land on both sides, so the model can partly memorize "how this brand's
pages behave" instead of learning something that transfers to a brand it hasn't seen. The paper also
doesn't print the base rate of "growing" in the holdout set next to the 71% accuracy number, so I
can't yet tell how many of those points are real skill versus the label's own class balance. Neither
gap invalidates the finding — the direction (younger, more-visible pages skew toward growth) is
plausible and matches Finding 1's descriptive framing — but both are checkable, and I'd ask for the
brand-grouped number and the base rate before repeating the 71% figure on its own.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_audit = pd.DataFrame([
    {
        "finding": "What Predicts Health? (Random Forest importance for health_score)",
        "reported_result": "Position 43%, Impressions 32%, Scroll Depth 15% importance",
        "where_label_comes_from": "health_score = impressions(30) + position(30) + ctr(20) + scroll_depth(20) -- a composite the paper itself defines on the Methodology page",
        "my_methodology_question": "Two of three top features are addends inside the label's own formula (Type-1 label-derived leakage) -- does importance add information beyond the scoring formula?",
        "paper_already_discloses": True,
    },
    {
        "finding": "What Predicts Growth? (Logistic Regression, 71% holdout accuracy)",
        "reported_result": "71% holdout accuracy; Content Age = strongest negative signal",
        "where_label_comes_from": "growing vs declining split on the 61.8K active-content sample, 80/20 holdout (Methodology page)",
        "my_methodology_question": "Is the 80/20 split random or grouped by the 57 brands? And what is the base rate of 'growing' in the holdout, so 71% can be read as skill-above-baseline?",
        "paper_already_discloses": False,
    },
])
pd.set_option("display.max_colwidth", None)
paper_audit

,finding,reported_result,where_label_comes_from,my_methodology_question,paper_already_discloses
0,What Predicts Health? (Random Forest importance for health_score),"Position 43%, Impressions 32%, Scroll Depth 15% importance",health_score = impressions(30) + position(30) + ctr(20) + scroll_depth(20) -- a composite the paper itself defines on the Methodology page,Two of three top features are addends inside the label's own formula (Type-1 label-derived leakage) -- does importance add information beyond the scoring formula?,True
1,"What Predicts Growth? (Logistic Regression, 71% holdout accuracy)",71% holdout accuracy; Content Age = strongest negative signal,"growing vs declining split on the 61.8K active-content sample, 80/20 holdout (Methodology page)","Is the 80/20 split random or grouped by the 57 brands? And what is the base rate of 'growing' in the holdout, so 71% can be read as skill-above-baseline?",False


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 notebook only ever trained under the client-grouped split — I never actually showed
the "before" number, so I couldn't point to the gap. Here I rebuild the exact same feature set,
same label, same Random Forest, and compare two splits side by side:

- **Before (naive):** plain random 70/30 row-level split, `train_test_split(..., stratify=label)` —
  rows from the same client can land on both sides.
- **After (honest):** the same `GroupShuffleSplit` by `client_id` I used in Week 5 — a client is
  entirely in train or entirely in test, never both.

Both use the same seed, same test proportion, same model hyperparameters, so the split is the only
thing that changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
has_search = work_has_features = None  # placeholder cleared below

work = df.copy()
work["has_search_volume_data"] = work["search_volume"].notna().astype(int)
work["has_word_count_data"] = work["word_count"].notna().astype(int)
work["has_position_data"] = (work["avg_position"] > 0).astype(int)

label = (work["trend_direction"] == "down").astype(int)

# Same honest candidate list as w05_model / w03_feature_leakage_check (leaky sibling windows
# impressions_last_30d/prev_30d + clicks/sessions siblings already excluded, confirmed leaky in w03)
numeric_candidates = [
    "content_age_days", "days_since_last_update", "word_count", "char_count",
    "search_volume", "competition", "cpc",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
X_numeric = work[numeric_candidates].fillna(0)
X_numeric = pd.concat(
    [X_numeric, work[["has_search_volume_data", "has_word_count_data", "has_position_data"]]],
    axis=1,
)
categorical_candidates = ["content_type", "main_intent", "competition_level"]
X_categorical = pd.get_dummies(work[categorical_candidates], dummy_na=True, prefix=categorical_candidates)

X = pd.concat([X_numeric, X_categorical], axis=1)
groups = work["client_id"]

print("Feature matrix shape:", X.shape)


def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()


def fit_eval_rf(X_train, X_test, y_train, y_test):
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    y_test_r = y_test.reset_index(drop=True)
    p50 = precision_at_k(proba, y_test_r, 50)
    return rf, proba, auc, p50


# --- BEFORE: naive random 70/30 split (row-level, ignores client_id) ---
Xtr_rand, Xte_rand, ytr_rand, yte_rand = train_test_split(
    X, label, test_size=0.3, random_state=RANDOM_SEED, stratify=label
)
rf_rand, proba_rand, auc_rand, p50_rand = fit_eval_rf(Xtr_rand, Xte_rand, ytr_rand, yte_rand)

# client overlap check for the naive split (train_test_split doesn't know client_id exists)
groups_train_idx = Xtr_rand.index
groups_test_idx = Xte_rand.index
overlap_rand = set(groups.loc[groups_train_idx]) & set(groups.loc[groups_test_idx])

# --- AFTER: honest split, grouped by client_id (same as w05_model) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, label, groups))
Xtr_grp, Xte_grp = X.iloc[train_idx], X.iloc[test_idx]
ytr_grp, yte_grp = label.iloc[train_idx], label.iloc[test_idx]
rf_grp, proba_grp, auc_grp, p50_grp = fit_eval_rf(Xtr_grp, Xte_grp, ytr_grp, yte_grp)
overlap_grp = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])

print(f"{'Split':<28}{'client overlap':>16}{'AUC':>10}{'precision@50':>16}")
print(f"{'Before -- random 70/30':<28}{len(overlap_rand):>16}{auc_rand:>10.3f}{p50_rand:>16.3f}")
print(f"{'After -- grouped by client':<28}{len(overlap_grp):>16}{auc_grp:>10.3f}{p50_grp:>16.3f}")
print(f"\nBase rate (full data): {round(label.mean(), 3)}")
print(f"AUC gap (random minus grouped): {round(auc_rand - auc_grp, 3)}")

Feature matrix shape: (30000, 38)
Split                         client overlap       AUC    precision@50
Before -- random 70/30                    31     0.754           0.980
After -- grouped by client                 0     0.763           0.840

Base rate (full data): 0.542
AUC gap (random minus grouped): -0.01


**Result (numbers from the code cell above).** The naive random split lets 31 clients appear on
both sides of train/test; the grouped split has zero overlap by construction, as designed.

AUC barely moved (0.754 naive vs 0.763 grouped — a 0.01 gap, and in the direction I did NOT expect:
the naive split did not look inflated on AUC). But **precision@50 tells a different story**: 0.980
on the naive split vs 0.840 on the grouped split — a 0.14 gap. My read: AUC is a whole-ranking
metric averaged over 9,000 test rows, so a small amount of per-client memorization from 31 leaked
clients barely moves it. Precision@50 only looks at the very top of the ranked list, and it looks
like some of those top-50 slots on the naive split were pages whose *client* the model had partly
memorized rather than pages it correctly judged declining on their own merits — exactly the
scenario the client-grouped split exists to catch. This is the actual finding I'd flag if reviewing
my own work: **the metric I'd naturally headline (AUC) barely showed the problem; the metric that
matches my real deployment question (precision@50, an editor's top-of-queue picks) showed it
clearly.** The grouped number (0.840) is the one I trust and the one I already reported in Week 5.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Week 3 caught and removed the two sibling-window leaks (`impressions_last_30d` /
`impressions_prev_30d`). This section re-runs the attack checklist against the **final 38-column
feature set actually used in Week 5 and Section 2 above** — not to redo Week 3's work, but to
confirm nothing new crept back in and that the honest split doesn't hide a new problem.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("ATTACK CHECKLIST -- final feature set")
print("=" * 60)

# 1. Timeline / no label family columns in X
label_family = {"trend_direction", "trend_pct"}
overlap_label = label_family & set(X.columns)
print(f"[1] Label-family columns present in X: {sorted(overlap_label) or 'none'}")

# 2. No known-leaky sibling windows re-added
known_leaky = {"impressions_last_30d", "impressions_prev_30d",
               "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"}
overlap_leaky = known_leaky & set(X.columns)
print(f"[2] Known-leaky sibling windows present in X: {sorted(overlap_leaky) or 'none'}")

# 3. No pseudonymous IDs used as features (grouping only)
id_cols = {"content_id", "client_id"}
overlap_ids = id_cols & set(X.columns)
print(f"[3] ID columns present in X (should be grouping-only): {sorted(overlap_ids) or 'none'}")

# 4. No product/decision-derived flags (this dataset has none in the dictionary, confirm)
print(f"[4] Columns containing 'score' or 'flag' in X: "
      f"{[c for c in X.columns if 'score' in c.lower() or 'flag' in c.lower()] or 'none'}")

# 5. Base rate printed next to metrics (from Section 2 output)
print(f"[5] Base rate (full data): {round(label.mean(), 3)}  |  "
      f"grouped-split test base rate: {round(yte_grp.mean(), 3)}")

# 6. Top feature importance sanity check on the grouped-split model
importances_grp = pd.Series(rf_grp.feature_importances_, index=X.columns).sort_values(ascending=False)
print(f"\n[6] Top 5 feature importances (grouped-split model):")
print(importances_grp.head(5).to_string())
print(f"    Max single-feature importance: {importances_grp.max():.3f}  "
      f"(>0.5 would be a 'too good' flag worth investigating -- not the case here)")

# 7. Deliberate leak-injection: re-add a KNOWN leaky column and confirm the harness catches it
X_with_leak = X.copy()
X_with_leak["impressions_last_30d"] = work["impressions_last_30d"]
Xtr_leak, Xte_leak, ytr_leak, yte_leak = train_test_split(
    X_with_leak, label, test_size=0.3, random_state=RANDOM_SEED, stratify=label
)
_, _, auc_leak, _ = fit_eval_rf(Xtr_leak, Xte_leak, ytr_leak, yte_leak)
print(f"\n[7] Harness check -- AUC WITH known-leaky column re-added: {auc_leak:.3f} "
      f"(honest AUC without it: {auc_grp:.3f})")
print(f"    Jump of {round(auc_leak - auc_grp, 3)} confirms the test harness correctly detects "
      f"a real leak when one is present -- the earlier 'no leak' readings above are trustworthy, "
      f"not just an artifact of a harness that can't tell the difference.")

ATTACK CHECKLIST -- final feature set
[1] Label-family columns present in X: none
[2] Known-leaky sibling windows present in X: none
[3] ID columns present in X (should be grouping-only): none
[4] Columns containing 'score' or 'flag' in X: none
[5] Base rate (full data): 0.542  |  grouped-split test base rate: 0.599

[6] Top 5 feature importances (grouped-split model):
days_with_impressions    0.142080
avg_position             0.139647
impressions_90d          0.133234
content_age_days         0.102162
word_count               0.056973
    Max single-feature importance: 0.142  (>0.5 would be a 'too good' flag worth investigating -- not the case here)

[7] Harness check -- AUC WITH known-leaky column re-added: 0.802 (honest AUC without it: 0.763)
    Jump of 0.039 confirms the test harness correctly detects a real leak when one is present -- the earlier 'no leak' readings above are trustworthy, not just an artifact of a harness that can't tell the difference.


**Result.** No label-family columns, no known-leaky sibling windows, and no ID columns made it
into the final 38-column feature set — all clean. No product/decision-derived flags exist in this
dataset to begin with (confirmed against `docs/data-dictionary.md`), so that leakage type doesn't
apply here. The top feature importance (grouped-split model) tops out well below the "too good"
threshold, consistent with the Week-5 read (no single feature dominates the way the sibling-window
leak did in Week 3). Re-adding a known-leaky column on purpose (step 7) pushes AUC up sharply,
which confirms the harness itself is sensitive enough that its earlier "clean" readings mean
something — an honest audit needs a test that could have failed and didn't, not just an absence of
alarms.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from `w05_model.ipynb`, Section 4):**

> "Random Forest wins on AUC (0.763 vs LR's 0.725) and ties LR on precision@50 (0.84), so it's my
> pick — with the caveat that at very small k the plain baseline rule is still the safer, ship-it-
> today choice."

This is close to fine, but "wins" and "so it's my pick" read like a settled verdict rather than a
measured, single-holdout comparison — and the AUC numbers were from the client-grouped split I ran
once, not a number I've shown is stable across resamples.

**Rewrite (safe language: observed, measured, decision-support):**

> On this client-grouped holdout, Random Forest measured a higher AUC than Logistic Regression
> (0.763 vs 0.725) and matched it on precision@50 (0.84 vs 0.84). Based on this single split, Random
> Forest is my working choice for further testing — not a proven superior method, since I haven't
> re-run this comparison across multiple grouped splits to see how much the 0.038 AUC gap moves
> around. For decision support today: at very small k, the plain baseline rule (Week 4) is still the
> safer, ship-it-now option, since it doesn't depend on a held-out sample I've only checked once.

The rewrite keeps the same practical recommendation but drops "wins" (implies proof) and "so it's my
pick" (implies the decision is closed) in favor of "measured... on this split" and "working choice
for further testing" — language a reader can't over-trust past what one holdout run actually showed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claim_rewrite = pd.DataFrame([
    {"version": "original", "text": "Random Forest wins on AUC (0.763 vs LR's 0.725) and ties LR on precision@50 (0.84), so it's my pick."},
    {"version": "rewritten", "text": "On this client-grouped holdout, Random Forest measured a higher AUC than Logistic Regression (0.763 vs 0.725) and matched it on precision@50. Based on this single split, it is my working choice for further testing, not a proven superior method."},
])
claim_rewrite

,version,text
0,original,"Random Forest wins on AUC (0.763 vs LR's 0.725) and ties LR on precision@50 (0.84), so it's my pick."
1,rewritten,"On this client-grouped holdout, Random Forest measured a higher AUC than Logistic Regression (0.763 vs 0.725) and matched it on precision@50. Based on this single split, it is my working choice for further testing, not a proven superior method."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.